In [1]:
# notebook : semdetect_e2.ipynb — cellule 0 : imports et configuration
import json
import logging
import os
from pathlib import Path
import time
import numpy as np
import pandas as pd
from transformers import AutoTokenizer

from src.data.dataset_semdetect import SemDetectDataset

logging.basicConfig(level=logging.INFO)
logging.getLogger("httpx").setLevel(logging.WARNING)

CACHE_ROOT = Path(os.environ["CACHE_DIR"])
REVISION = "18b74b551670bff6b04e57570232a1694997863b"

GEN_MAX_TOKENS = 3072        # review_generator.generate_ai_review
PROMPT_OVERHEAD = 400        # prompt système + "Paper Content:" + gabarit de chat (estimation haute)

/home/cyaiche/generated_text_detection_hub/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-10-05 03:09:20.712 | INFO     | src.config:<module>:12 - PROJ_ROOT path is: /home/cyaiche/generated_text_detection_hub


In [2]:
# notebook : semdetect_e2.ipynb — cellule 1 : dimensionnement des générations et longueur des papiers
te = SemDetectDataset(config={
    "name": "sem_detect", "repo_id": "Sem-Detect/ML_Conferences-Peer-Reviews", "revision": REVISION,
    "split": "test", "conference": None, "date": None, "generator": None,
    "label_mode": "three_class", "balance": "none", "subset_size": None,
    "val_fraction": None, "val_seed": 0,
    "no_duplicate": True, "keep_meta": True, "with_paper": True, "keep_claims": False,
    "exclude_ids_path": None, "seed": 0,
}).to_pandas()

# ---- volume de génération : un jeu de 3 références par couple (papier, note) ----
pairs = te.groupby(["submission_id", "rating"]).size()
print(f"{len(te)} cibles | {te['submission_id'].nunique()} papiers | {len(pairs)} couples (papier, note)")
print(f"=> {3 * len(pairs)} générations de référence (3 modèles)")
print("répartition des notes sur les couples :", pairs.index.get_level_values("rating").value_counts().to_dict(), "\n")

# ---- longueur des papiers ----
papers = te.drop_duplicates("submission_id")[["submission_id", "conference", "year", "paper_content", "paper_appendix"]].copy()
print(f"papiers sans texte : {(papers['paper_content'].str.strip() == '').sum()} | "
      f"avec annexe : {(papers['paper_appendix'].str.strip() != '').sum()} / {len(papers)}")

tok = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")
n_tokens = lambda t: len(tok(t, add_special_tokens=False)["input_ids"]) if t else 0
papers["tok_main"] = papers["paper_content"].map(n_tokens)
papers["tok_full"] = (papers["paper_content"] + "\n\n" + papers["paper_appendix"]).map(n_tokens)

print("\nTokens par papier (tokenizer Mistral-7B, estimation) :")
print(papers[["tok_main", "tok_full"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(0).T, "\n")

print("Part des papiers qui tiennent dans le contexte (papier + prompt + 3072 tokens de sortie) :")
rows = []
for ctx in (16384, 24576, 32768, 65536):
    budget = ctx - GEN_MAX_TOKENS - PROMPT_OVERHEAD
    rows.append({"contexte": ctx, "budget papier": budget,
                 "corps seul": (papers["tok_main"] <= budget).mean(),
                 "corps + annexes": (papers["tok_full"] <= budget).mean()})
print(pd.DataFrame(rows).set_index("contexte").round(3))

INFO:src.data.dataset_semdetect:📦 Loading Sem-Detect dataset from cache: /data2/cyaiche/cache_dir/dataset_cache/dataset_d29debee44bd4f53619a40fc7c54250b.pt


4410 cibles | 159 papiers | 340 couples (papier, note)
=> 1020 générations de référence (3 modèles)
répartition des notes sur les couples : {'Borderline': 141, 'Weak Accept': 80, 'Accept': 68, 'Weak Reject': 28, 'Reject': 23} 

papiers sans texte : 0 | avec annexe : 143 / 159

Tokens par papier (tokenizer Mistral-7B, estimation) :
          count     mean      std     min      50%      90%      95%      99%  \
tok_main  159.0  14529.0   2715.0  8035.0  14428.0  17326.0  18376.0  20799.0   
tok_full  159.0  20085.0  10378.0  8556.0  16966.0  30997.0  36267.0  47296.0   

               max  
tok_main   30872.0  
tok_full  110377.0   

Part des papiers qui tiennent dans le contexte (papier + prompt + 3072 tokens de sortie) :
          budget papier  corps seul  corps + annexes
contexte                                            
16384             12912       0.277            0.094
24576             21104       0.987            0.711
32768             29296       0.994            0.881
65

In [3]:
# notebook : semdetect_e2.ipynb — cellule 2 : une génération par couple (papier, note) et par modèle
from src.detector.sem_detect.claims_llm import LocalClaimExtractor
from src.detector.sem_detect.llm_jobs import LocalReviewCleaner, LocalReviewGenerator
from src.utils.vllm_manager import VLLMManager

GENERATORS = {  # modèle servi -> profil vLLM
    "mistral-small-3.2-fp8": "mistral-small-3.2-batch",
    "gemma-3-27b": "gemma-3-27b-batch",
    "glm-4-32b": "glm-4-32b-batch",
}

E2_DIR = CACHE_ROOT / "sem_detect_e2"
E2_DIR.mkdir(parents=True, exist_ok=True)

pairs_df = te.groupby(["submission_id", "rating"], as_index=False).size()[["submission_id", "rating"]]
paper_body = te.drop_duplicates("submission_id").set_index("submission_id")["paper_content"]  # corps seul
gen_items = [{"id": f"{s}|{r}", "paper_text": paper_body[s], "score": r}
             for s, r in zip(pairs_df["submission_id"], pairs_df["rating"])]
print(f"{len(gen_items)} couples x {len(GENERATORS)} modèles = {len(gen_items) * len(GENERATORS)} générations")

340 couples x 3 modèles = 1020 générations


In [4]:
# notebook : semdetect_e2.ipynb — cellule 3 (remplace la précédente) : génération, modèles terminés sautés
mgr = VLLMManager(gpu=1, restart_watchdog=True)
gen_status = {}
for served, profile in GENERATORS.items():
    try:
        gen = LocalReviewGenerator(cache_root=CACHE_ROOT, served_model=served, max_workers=8)
        if (gen.collect([it["id"] for it in gen_items])["status"] == "ok").all():
            print(f"{served} : cache complet, serveur non démarré")
        else:
            if mgr.profile_served_name(profile) != served:
                raise ValueError(f"SERVED_NAME du profil {profile!r} != {served!r}")
            mgr.ensure(profile)
            gen.run(gen_items)
        out = gen.collect([it["id"] for it in gen_items])
        ok = out[out["status"] == "ok"]
        gen_status[served] = {
            "ok": len(ok),
            "erreurs": int((out["status"] != "ok").sum()),
            "papiers tronqués": int(ok["truncated"].fillna(False).astype(bool).sum()) if "truncated" in ok else 0,
            "sortie coupée à 3072": int((ok["finish_reason"] == "length").sum()) if "finish_reason" in ok else 0,
            "tokens sortie (moy.)": round(float(ok["completion_tokens"].mean()), 0) if len(ok) else None,
            "s / génération (moy.)": round(float(ok["seconds"].mean()), 1) if len(ok) else None,
            "source": gen.source_tag,
        }
    except Exception as e:
        gen_status[served] = {"échec": f"{type(e).__name__}: {e}"[:400]}
        print(f"\n!!! {served} : {gen_status[served]['échec']}\n")

print(pd.DataFrame(gen_status).T.to_string())

INFO:src.detector.sem_detect.llm_jobs:review_generation [mistral-small-3.2-fp8] cache review_generation_mistral-small-3.2-fp8_73e064dc47.jsonl : 340 ok
INFO:src.detector.sem_detect.llm_jobs:review_generation [gemma-3-27b] cache review_generation_gemma-3-27b_3df521d10e.jsonl : 340 ok
INFO:src.detector.sem_detect.llm_jobs:review_generation [glm-4-32b] cache review_generation_glm-4-32b_29b03f94c9.jsonl : 340 ok


mistral-small-3.2-fp8 : cache complet, serveur non démarré
gemma-3-27b : cache complet, serveur non démarré
glm-4-32b : cache complet, serveur non démarré
                        ok erreurs papiers tronqués sortie coupée à 3072 tokens sortie (moy.) s / génération (moy.)                                              source
mistral-small-3.2-fp8  340       0                0                    0                705.0                 113.7  review_generation:mistral-small-3.2-fp8:73e064dc47
gemma-3-27b            340       0                0                    0                714.0                  90.6            review_generation:gemma-3-27b:3df521d10e
glm-4-32b              340       0                2                    0                775.0                 120.9              review_generation:glm-4-32b:29b03f94c9


In [5]:
# notebook : semdetect_e2.ipynb — cellule 4 : nettoyage puis claims des références (Qwen3-32B, GPU 1)
mgr.ensure("qwen3-32b-batch")

raw_refs = []
for served in GENERATORS:
    out = LocalReviewGenerator(cache_root=CACHE_ROOT, served_model=served).collect([it["id"] for it in gen_items])
    for rec in out[out["status"] == "ok"].to_dict("records"):
        raw_refs.append({"id": f"{served}|{rec['id']}", "generator": served, "pair": rec["id"], "text": rec["text"]})
print(f"{len(raw_refs)} références brutes à nettoyer")

cleaner = LocalReviewCleaner(cache_root=CACHE_ROOT, served_model="qwen3-32b", max_workers=32)
cleaned = cleaner.run(raw_refs).set_index("id")

refs_df = pd.DataFrame(raw_refs).set_index("id")
refs_df["clean_ok"] = cleaned["status"].eq("ok")
# même repli que clean_review d'origine : texte brut si le nettoyage échoue
refs_df["content"] = np.where(refs_df["clean_ok"], cleaned["text"], refs_df["text"])
refs_df["braces"] = cleaned.get("braces")
refs_df["review-id"] = "ref:" + refs_df.index
refs_df[["submission_id", "rating"]] = refs_df["pair"].str.split("|", n=1, expand=True)

print(f"nettoyage : {refs_df['clean_ok'].mean():.1%} ok | accolades trouvées : {refs_df['braces'].fillna(False).mean():.1%}")
print("longueur nettoyée / brute (médiane, par modèle) :")
print((refs_df["content"].str.len() / refs_df["text"].str.len()).groupby(refs_df["generator"]).median().round(3))

extractor = LocalClaimExtractor(cache_root=CACHE_ROOT, served_model="qwen3-32b", max_workers=32)
ref_claims = extractor.extract(refs_df[["review-id", "content"]])
ref_claims = extractor.retry_failed(refs_df[["review-id", "content"]], max_tokens=8192)
ref_claims = extractor.retry_failed(refs_df[["review-id", "content"]], max_tokens=8192,
                                    extra_sampling={"repetition_penalty": 1.05})
print("claims :", ref_claims["claims_status"].value_counts().to_dict())

refs_df = refs_df.reset_index().merge(ref_claims[["review-id", "claim_extraction", "claims_status"]], on="review-id")
refs_df.drop(columns=["text"]).to_parquet(E2_DIR / "local_refs.parquet", index=False)
print(f"sauvegardé : {E2_DIR / 'local_refs.parquet'} ({len(refs_df)} références)")

INFO:src.utils.vllm_manager:qwen3-32b déjà servi et opérationnel : réutilisé
INFO:src.detector.sem_detect.llm_jobs:review_generation [mistral-small-3.2-fp8] cache review_generation_mistral-small-3.2-fp8_73e064dc47.jsonl : 340 ok
INFO:src.detector.sem_detect.llm_jobs:review_generation [gemma-3-27b] cache review_generation_gemma-3-27b_3df521d10e.jsonl : 340 ok
INFO:src.detector.sem_detect.llm_jobs:review_generation [glm-4-32b] cache review_generation_glm-4-32b_29b03f94c9.jsonl : 340 ok
INFO:src.detector.sem_detect.llm_jobs:review_cleaning [qwen3-32b] cache review_cleaning_qwen3-32b_2ac8859781.jsonl : 1020 ok


1020 références brutes à nettoyer
nettoyage : 100.0% ok | accolades trouvées : 100.0%
longueur nettoyée / brute (médiane, par modèle) :
generator
gemma-3-27b              0.801
glm-4-32b                0.877
mistral-small-3.2-fp8    0.745
dtype: float64


INFO:src.detector.sem_detect.claims_llm:Claims cache /data2/cyaiche/cache_dir/claims_llm/vllm_qwen3-32b_0902bb5d60.jsonl : 5430 ok
INFO:src.detector.sem_detect.claims_llm:relance de 0 échec(s) avec max_tokens=8192, extra=None
INFO:src.detector.sem_detect.claims_llm:relance de 0 échec(s) avec max_tokens=8192, extra={'repetition_penalty': 1.05}


claims : {'ok': 1020}
sauvegardé : /data2/cyaiche/cache_dir/sem_detect_e2/local_refs.parquet (1020 références)


In [6]:
# notebook : semdetect_e2.ipynb — cellule 5 : caches, références locales et classifieur
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support, roc_auc_score

from src.config import PROJ_ROOT
from src.detector.sem_detect.feature_store import SemDetectFeatureStore, parse_claims
from src.detector.sem_detect.references import (
    build_feature_matrix, fixed_models_references, paper_protocol_references, predict_proba,
)
from src.detector.sem_detect.semdetect.claim_extractor import SEMANTIC_CATEGORIES
from src.detector.sem_detect.semdetect.classifier import FEATURE_DISPLAY_NAMES, SemDetectClassifier
from src.evaluation.metrics import get_tpr_at_fpr

STORE_ROOT = CACHE_ROOT / "sem_detect_store"
HF_SOURCE = f"hf:Sem-Detect/ML_Conferences-Peer-Reviews@{REVISION}"
TEXT_SOURCE = f"{HF_SOURCE}:clean_review"
FEATURES_DEVICE = "cuda:0"

extractor = LocalClaimExtractor(cache_root=CACHE_ROOT, served_model="qwen3-32b")
store_e0 = SemDetectFeatureStore(root=STORE_ROOT, claims_source=HF_SOURCE, text_source=TEXT_SOURCE,
                                 device=FEATURES_DEVICE)
store_loc = SemDetectFeatureStore(root=STORE_ROOT, claims_source=extractor.source_tag, text_source=TEXT_SOURCE,
                                  device=FEATURES_DEVICE)  # claims Qwen3-32B : cibles (E1) + références locales (E2)

refs_df = pd.read_parquet(E2_DIR / "local_refs.parquet")
assert (refs_df["claims_status"] == "ok").all(), "références sans claims"
store_loc.ensure_embeddings(refs_df[["review-id", "claim_extraction"]])  # ~5 min, GPU 0


def has_embedding(store: SemDetectFeatureStore, rid: str) -> bool:
    try:
        store.semantic(rid, 0)
        return True
    except KeyError:
        return False


missing_targets = [r for r in te["review-id"] if not has_embedding(store_loc, r)]
missing_refs = [r for r in refs_df["review-id"] if not has_embedding(store_loc, r)]
print(f"embeddings manquants : cibles {len(missing_targets)} | références {len(missing_refs)} (attendu : 0 / 0)")

clf = SemDetectClassifier(str(PROJ_ROOT / "src/detector/sem_detect/models/lightgbm_best_model.pkl"))
te_idx = te.set_index("review-id")

INFO:src.detector.sem_detect.claims_llm:Claims cache /data2/cyaiche/cache_dir/claims_llm/vllm_qwen3-32b_0902bb5d60.jsonl : 5430 ok
INFO:src.detector.sem_detect.feature_store:Feature store : 4410 embeddings (0295060840ff), 4410 stats textuelles (2af061950803)
INFO:src.detector.sem_detect.feature_store:Feature store : 4410 embeddings (fbbbe1608d0e), 4410 stats textuelles (2af061950803)
Embeddings (fragments):   0%|          | 0/4 [00:00<?, ?it/s]INFO:src.detector.sem_detect.semdetect.embedding_generator:Embedding device: cuda:0
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from Qwen/Qwen3-Embedding-0.6B.

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 312.54it/s]
INFO:sentence_transformers.base.model:Loaded 1 prompt with these keys: ['query']
INFO:src.detector.sem_detect.semdetect.embedding_generator:Embedding model loaded

Embeddings (fragments): 100%|██████████| 4/4 [00:21<00:00,  5.47s/it]
INFO:src.detector.sem_detect.semdetect.classifier:Loading Sem

embeddings manquants : cibles 0 | références 0 (attendu : 0 / 0)


INFO:src.detector.sem_detect.semdetect.classifier:SHAP TreeExplainer initialized
INFO:src.detector.sem_detect.semdetect.classifier:Model loaded (similarity_threshold=0.80)


In [7]:
# notebook : semdetect_e2.ipynb — cellule 6 : les trois jeux de références
def valid_in(store: SemDetectFeatureStore):
    return lambda rid: store.semantic(rid, 1) is not None


refs_e0 = paper_protocol_references(te, k=3, seed=0, is_valid=valid_in(store_e0))   # dataset, claims Gemini
refs_e1 = paper_protocol_references(te, k=3, seed=0, is_valid=valid_in(store_loc))  # dataset, claims Qwen3-32B
refs_e2 = fixed_models_references(te, refs_df, models=list(GENERATORS), is_valid=valid_in(store_loc))  # locales

print("nombre de références locales par cible (E2) :",
      pd.Series([len(v) for v in refs_e2.values()]).value_counts().sort_index().to_dict())

nombre de références locales par cible (E2) : {0: 23, 1: 108, 2: 1095, 3: 3184}


In [8]:
# notebook : semdetect_e2.ipynb — cellule 7 : E0 / E1 / E2, mêmes cibles, même classifieur
CLASS_TO_IDX = {"ai": 0, "rewrite": 1, "human": 2}
NAMES = ["AI", "LLM-Refined", "Human"]


def run(store: SemDetectFeatureStore, refs_map: dict):
    X, kept, skipped = build_feature_matrix(store, te["review-id"].tolist(), refs_map, clf.similarity_threshold)
    P = pd.DataFrame(predict_proba(clf, X), index=kept, columns=["p_ai", "p_rewrite", "p_human"])
    return P, pd.DataFrame(X, index=kept, columns=FEATURE_DISPLAY_NAMES), skipped


def metrics(P: pd.DataFrame) -> dict:
    cls = te_idx.loc[P.index, "class"]
    y, yp = cls.map(CLASS_TO_IDX).to_numpy(), P.to_numpy().argmax(axis=1)
    prec, rec, _, _ = precision_recall_fscore_support(y, yp, labels=[0, 1, 2], zero_division=0)
    p_ai, is_ai = P["p_ai"].to_numpy(), (cls == "ai").to_numpy()
    m = cls.isin(["human", "ai"]).to_numpy()
    return {
        "n classées": len(y), "macro-F1": f1_score(y, yp, average="macro"),
        "P_ai": prec[0], "R_ai": rec[0], "P_rewrite": prec[1], "R_rewrite": rec[1], "P_human": prec[2], "R_human": rec[2],
        "AUC humain/IA": roc_auc_score(is_ai[m], p_ai[m]), "TPR@1% humain/IA": get_tpr_at_fpr(is_ai[m], p_ai[m], 0.01),
        "AUC (h+r)/IA": roc_auc_score(is_ai, p_ai),
    }


def paired_delta(Pa: pd.DataFrame, Pb: pd.DataFrame, n_boot: int = 1000, seed: int = 0):
    """macro-F1(b) − macro-F1(a) sur les cibles classées dans les deux, IC95 bootstrap."""
    common = Pa.index.intersection(Pb.index)
    y = te_idx.loc[common, "class"].map(CLASS_TO_IDX).to_numpy()
    ya, yb = Pa.loc[common].to_numpy().argmax(axis=1), Pb.loc[common].to_numpy().argmax(axis=1)
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        diffs.append(f1_score(y[i], yb[i], average="macro") - f1_score(y[i], ya[i], average="macro"))
    point = f1_score(y, yb, average="macro") - f1_score(y, ya, average="macro")
    return len(common), point, *np.percentile(diffs, [2.5, 97.5]), (ya == yb).mean()


results = {
    "E0 Gemini + réf. dataset": run(store_e0, refs_e0),
    "E1 Qwen + réf. dataset": run(store_loc, refs_e1),
    "E2 Qwen + réf. locales": run(store_loc, refs_e2),
}

print("=== Métriques ===")
print(pd.DataFrame({name: metrics(P) for name, (P, _, _) in results.items()}).round(4), "\n")

print("=== Cibles non classables ===")
for name, (_, _, skipped) in results.items():
    s = pd.Series(skipped, dtype=object)
    by = s.to_frame("raison").join(te_idx["class"]).value_counts().to_dict() if len(s) else {}
    print(f"{name} : {len(s)} {by}")
print()

print("=== Confusion (% par ligne vraie) ===")
for name, (P, _, _) in results.items():
    cls = te_idx.loc[P.index, "class"]
    cm = confusion_matrix(cls.map(CLASS_TO_IDX), P.to_numpy().argmax(axis=1), labels=[0, 1, 2], normalize="true")
    print(f"{name}\n{pd.DataFrame(np.round(cm * 100, 2), index=NAMES, columns=NAMES)}\n")

print("=== Comparaisons appariées (Δ macro-F1, IC95 bootstrap) ===")
P0, P1, P2 = (results[k][0] for k in results)
for label, (Pa, Pb) in {"E2 − E1 (effet des références)": (P1, P2),
                        "E2 − E0 (extracteur + références)": (P0, P2)}.items():
    n, d, lo, hi, agree = paired_delta(Pa, Pb)
    print(f"{label:<36} n={n}  Δ={d:+.4f}  [{lo:+.4f}, {hi:+.4f}]  accord={agree:.1%}")

=== Métriques ===
                  E0 Gemini + réf. dataset  E1 Qwen + réf. dataset  \
n classées                       4373.0000               4366.0000   
macro-F1                            0.8418                  0.8258   
P_ai                                0.9381                  0.9590   
R_ai                                0.9113                  0.8434   
P_rewrite                           0.8725                  0.8389   
R_rewrite                           0.9232                  0.9473   
P_human                             0.7751                  0.8027   
R_human                             0.6444                  0.6054   
AUC humain/IA                       0.9980                  0.9963   
TPR@1% humain/IA                    0.9282                  0.9272   
AUC (h+r)/IA                        0.9912                  0.9902   

                  E2 Qwen + réf. locales  
n classées                     3165.0000  
macro-F1                          0.7345  
P_ai        

In [9]:
# notebook : semdetect_e2.ipynb — cellule 8 : diagnostics des références locales
def n_semantic(raw: str) -> int:
    c = parse_claims(raw)
    return sum(len(getattr(c, cat)) for cat in SEMANTIC_CATEGORIES)


# a) longueur en claims : références locales vs reviews du dataset, même extracteur (Qwen3-32B)
local_counts = refs_df.assign(n=refs_df["claim_extraction"].map(n_semantic)).groupby("generator")["n"]
te_claims = te[["review-id", "class", "generator"]].merge(extractor.collect(te), on="review-id")
te_claims = te_claims[te_claims["claims_status"] == "ok"].assign(n=lambda d: d["claim_extraction"].map(n_semantic))
print("Claims sémantiques par review (extracteur Qwen3-32B) :")
print(pd.concat([
    local_counts.agg(["mean", "median"]).rename(index=lambda g: f"local / {g}"),
    te_claims.groupby(["class", "generator"])["n"].agg(["mean", "median"])
              .rename(index=lambda t: f"dataset / {t[0]} / {t[1]}"),
]).round(1), "\n")

# b) mécanisme central : similarité de meilleur appariement par classe (Figure 6 du papier)
feat = "Mean best-match similarity"
rows = {}
for name, (_, X, _) in results.items():
    med = X[feat].groupby(te_idx.loc[X.index, "class"]).median()
    rows[name] = {**med.to_dict(), "écart IA − humain": med["ai"] - med["human"]}
print(f"{feat} — médiane par classe :")
print(pd.DataFrame(rows).T.round(4), "\n")

# c) rappel IA selon le modèle qui a produit la cible
print("Rappel IA par générateur de la cible :")
recall = {}
for name, (P, _, _) in results.items():
    ai = P[te_idx.loc[P.index, "class"] == "ai"]
    recall[name] = (ai.to_numpy().argmax(axis=1) == 0).astype(float)
    recall[name] = pd.Series(recall[name], index=ai.index).groupby(te_idx.loc[ai.index, "generator"]).mean()
print(pd.DataFrame(recall).round(3), "\n")

# d) moyenne des 9 features sur les cibles communes
common = P0.index.intersection(P1.index).intersection(P2.index)
print("Moyenne des 9 features (cibles communes) :")
print(pd.DataFrame({"train (scaler)": clf.scaler.mean_,
                    **{name: X.loc[common].mean().to_numpy() for name, (_, X, _) in results.items()}},
                   index=FEATURE_DISPLAY_NAMES).round(4))

Claims sémantiques par review (extracteur Qwen3-32B) :
                                    mean  median
local / gemma-3-27b                 16.3    17.0
local / glm-4-32b                   13.4    14.0
local / mistral-small-3.2-fp8       11.0    13.0
(dataset / a / i, dataset / d / e)  14.5    15.0
(dataset / a / i, dataset / g / e)  17.1    17.0
(dataset / a / i, dataset / g / e)  15.0    14.0
(dataset / a / i, dataset / q / w)  17.1    17.0
(dataset / h / u, dataset / h / u)  12.8    12.0
(dataset / r / e, dataset / d / e)  12.7    12.0
(dataset / r / e, dataset / g / e)  13.4    12.0
(dataset / r / e, dataset / g / e)  13.4    13.0
(dataset / r / e, dataset / q / w)  14.1    13.0 

Mean best-match similarity — médiane par classe :
                              ai   human  rewrite  écart IA − humain
E0 Gemini + réf. dataset  0.7235  0.6321   0.6435             0.0915
E1 Qwen + réf. dataset    0.7380  0.6337   0.6343             0.1044
E2 Qwen + réf. locales    0.7114  0.6340   0.6434

In [10]:
# notebook : semdetect_e2.ipynb — cellule 9 : diagnostic des références sans claim sémantique
from src.detector.sem_detect.semdetect.claim_extractor import CLAIM_CATEGORIES

# a) quelles références sont invalides (cohérence entre le cache d'embeddings et les claims)
refs_df["n_sem"] = refs_df["claim_extraction"].map(n_semantic)
refs_df["invalid"] = refs_df["review-id"].map(lambda r: store_loc.semantic(r, 1) is None)
assert (refs_df["invalid"] == (refs_df["n_sem"] == 0)).all(), "incohérence entre embeddings et claims"
print("Références sans claim sémantique, par générateur :")
print(refs_df.groupby("generator")["invalid"].agg(invalides="sum", total="count", part="mean").round(3), "\n")

# b) textes bruts (cache de génération) et fin du nettoyage (cache de nettoyage)
raw_text, gen_finish = {}, {}
for served in GENERATORS:
    g = LocalReviewGenerator(cache_root=CACHE_ROOT, served_model=served)
    for rec in g.collect(sorted(refs_df.loc[refs_df["generator"] == served, "pair"].unique())).to_dict("records"):
        raw_text[f"{served}|{rec['id']}"] = rec.get("text") or ""
        gen_finish[f"{served}|{rec['id']}"] = rec.get("finish_reason")

clean_recs = LocalReviewCleaner(cache_root=CACHE_ROOT, served_model="qwen3-32b").collect(refs_df["id"].tolist()).set_index("id")
refs_df["raw_len"] = refs_df["id"].map(lambda i: len(raw_text.get(i, "")))
refs_df["clean_len"] = refs_df["content"].str.len()
refs_df["clean_ratio"] = refs_df["clean_len"] / refs_df["raw_len"].clip(lower=1)
refs_df["clean_finish"] = refs_df["id"].map(clean_recs["finish_reason"])
refs_df["gen_finish"] = refs_df["id"].map(gen_finish)

cats = pd.DataFrame([{c: len(getattr(parse_claims(r), c)) for c in CLAIM_CATEGORIES} for r in refs_df["claim_extraction"]],
                    index=refs_df.index)
print("Médianes par générateur et validité (longueurs en caractères, claims par catégorie) :")
print(refs_df.join(cats).groupby(["generator", "invalid"])[["raw_len", "clean_len", "clean_ratio", *CLAIM_CATEGORIES]]
      .median().round(2), "\n")

print("Fin de génération et fin de nettoyage, par générateur et validité :")
print(pd.crosstab([refs_df["generator"], refs_df["invalid"]], [refs_df["gen_finish"], refs_df["clean_finish"]]), "\n")

# c) un exemple invalide par générateur : début et fin du texte brut, du texte nettoyé, et claims
for served in GENERATORS:
    bad = refs_df[(refs_df["generator"] == served) & refs_df["invalid"]]
    if bad.empty:
        print(f"=== {served} : aucune référence invalide ===\n")
        continue
    ex = bad.iloc[0]
    raw = raw_text[ex["id"]]
    print(f"=== {served} | {ex['id']} | brut {len(raw)} car. -> nettoyé {len(ex['content'])} car. ===")
    print("--- brut, début ---\n" + raw[:600])
    print("--- brut, fin ---\n" + raw[-400:])
    print("--- nettoyé, début ---\n" + ex["content"][:600])
    print("--- nettoyé, fin ---\n" + ex["content"][-400:])
    print("--- claims ---\n" + json.dumps(json.loads(ex["claim_extraction"]), ensure_ascii=False, indent=1)[:1500])
    print()

INFO:src.detector.sem_detect.llm_jobs:review_generation [mistral-small-3.2-fp8] cache review_generation_mistral-small-3.2-fp8_73e064dc47.jsonl : 340 ok
INFO:src.detector.sem_detect.llm_jobs:review_generation [gemma-3-27b] cache review_generation_gemma-3-27b_3df521d10e.jsonl : 340 ok
INFO:src.detector.sem_detect.llm_jobs:review_generation [glm-4-32b] cache review_generation_glm-4-32b_29b03f94c9.jsonl : 340 ok


Références sans claim sémantique, par générateur :
                       invalides  total   part
generator                                     
gemma-3-27b                    7    340  0.021
glm-4-32b                     32    340  0.094
mistral-small-3.2-fp8         61    340  0.179 



INFO:src.detector.sem_detect.llm_jobs:review_cleaning [qwen3-32b] cache review_cleaning_qwen3-32b_2ac8859781.jsonl : 1020 ok


Médianes par générateur et validité (longueurs en caractères, claims par catégorie) :
                               raw_len  clean_len  clean_ratio  \
generator             invalid                                    
gemma-3-27b           False     3845.0     3112.0         0.80   
                      True      4000.0      636.0         0.15   
glm-4-32b             False     4094.5     3544.0         0.90   
                      True      4307.0      819.5         0.19   
mistral-small-3.2-fp8 False     3865.0     2943.0         0.77   
                      True      4152.0      788.0         0.19   

                               factualRestatement  evaluation  \
generator             invalid                                   
gemma-3-27b           False                   4.0        10.0   
                      True                    4.0         0.0   
glm-4-32b             False                   5.0         9.0   
                      True                    5.0         0.